# Data

In [2]:
import torch
import numpy as np

WINDOW_SIZE = 5
MIN_SEQ_LEN = 7
MAX_SEQ_LEN = 7
new_subj_Th = 10

DEV = 'cuda:1'

AE_FeatureExtract = 1

HORIZON = 4
ablation = 0
iteration = 8

AE_Hidden_Dim = 128

embed_Ftr_dim = 128


start_seq = MAX_SEQ_LEN - MIN_SEQ_LEN
end_seq = MAX_SEQ_LEN

N_age = 9
N_Urea = 100
N_SCRr = 5
N_eGFR = 200
N_eGFR_Class = 4


data = np.load(f"../filtered_data/dataset_win-{WINDOW_SIZE}_seq-{MIN_SEQ_LEN}-{MAX_SEQ_LEN}_horiz-{HORIZON}_subj-{new_subj_Th}_v3_fold-{iteration}.npz")

X_train = data['X_train']
y_train = data['y_train']
X_val = data['X_val']
y_val = data['y_val']
X_test = data['X_test']
y_test = data['y_test']
data.close()


In [3]:
n_mech = 11
n_drug = 71+26
n_ddi = 44

X_train[:,:,12: 12 + n_mech + n_drug + n_ddi][X_train[:,:, 12: 12 + n_mech + n_drug + n_ddi]>1] = 1
X_val[:,:,12: 12 + n_mech + n_drug + n_ddi][X_val[:,:, 12: 12 + n_mech + n_drug + n_ddi]>1] = 1
X_test[:,:,12: 12 + n_mech + n_drug + n_ddi][X_test[:,:, 12: 12 + n_mech + n_drug + n_ddi]>1] = 1


# Ablation

In [4]:
def remove_egfr_features(X):

    X0 = X.copy()

    scr = X0[:,-1,5]
    age = X0[:,-1,-2]
    sex = X0[:,-1,-1]

    
    if ablation == 3:
        X[:, start_seq:end_seq, 0:8] = 0
    if ablation == 4:
        X[:, start_seq:end_seq, 8:12] = 0
    if ablation == 5:
        X[:, start_seq:end_seq, 12: 12 + n_mech + n_drug] = 0
    if ablation == 6:
        X[:, start_seq:end_seq, 12 + n_mech + n_drug: 12 + n_mech + n_drug + n_ddi] = 0
    if ablation == 9:
        X[:, start_seq:end_seq, 12: 12 + n_mech + n_drug] = 0
        X[:, start_seq:end_seq, 12 + n_mech + n_drug: 12 + n_mech + n_drug + n_ddi] = 0
    if ablation == 10:
        X[:, start_seq:end_seq, -2:-1] = 0
    if ablation == 11:
        X[:, start_seq:end_seq, -1:] = 0
    
    # The following features were de-activated and are not used
    X[:, start_seq:end_seq, 6:8] = 0
    X[:, start_seq:end_seq, 12: 12 + n_mech] = 0
    X[:, start_seq:end_seq, 0:5] = 0
    X[:, start_seq:end_seq, 8:11] = 0

    # feature vector concatenation
    feat = np.concatenate([
            X[:, start_seq:end_seq, 0:6]/N_SCRr,       # SCr
            X[:, start_seq:end_seq, 6:7]/N_eGFR,       # eGFR value
            X[:, start_seq:end_seq, 7:8]/N_eGFR_Class, # eGFR class
            X[:, start_seq:end_seq, 8:12]/N_Urea,      # Urea
            X[:, start_seq:end_seq, 12: 12 + n_mech + n_drug],            # Drugs Mechanisms
            X[:, start_seq:end_seq, 12 + n_mech + n_drug: 12 + n_mech + n_drug + n_ddi],            # DDI types
            X[:, start_seq:end_seq, -2:-1]//N_age/10,      # Age
            X[:, start_seq:end_seq, -1:]               # Sex
    ], axis=2)

    print(feat.shape)
    return feat, scr, age, sex


X_train, scr_train, age_train, sex_train = remove_egfr_features(X_train)
X_val, scr_val, age_val, sex_val   = remove_egfr_features(X_val)
X_test, scr_test, age_test, sex_test  = remove_egfr_features(X_test)

num_features = X_train.shape[-1]
print(f"num_features: {num_features}")
print(f"y_train: {y_train.shape}")

(47935, 7, 166)
(6650, 7, 166)
(13134, 7, 166)
num_features: 166
y_train: (47935,)


In [5]:
def compute_egfr_tensor(scr, age, sex):

    scr = torch.tensor(scr, dtype=torch.float32)
    age = torch.as_tensor(age, dtype=torch.float32)#.to(dev)
    sex = torch.as_tensor(sex, dtype=torch.float32)#.to(dev)
    
    age = age.view(scr.shape)
    sex = sex.view(scr.shape)
    
    kappa = torch.where(sex==1, 0.7, 0.9)
    alpha = torch.where(sex==1, -0.241, -0.302)
    gender_mult = torch.where(sex==1, 1.012, 1.0)

    term1 = torch.minimum(scr / kappa, torch.ones_like(scr)) ** alpha
    term2 = torch.maximum(scr / kappa, torch.ones_like(scr)) ** -1.200
    term3 = 0.9938 ** age

    return 142 * term1 * term2 * term3 * gender_mult


def assign_class_tensor(egfr):
    cls = torch.zeros_like(egfr, dtype=torch.long)

    cls[egfr >= 90] = 0
    cls[(egfr >= 60) & (egfr < 90)] = 1
    cls[(egfr >= 30) & (egfr < 60)] = 2
    cls[(egfr >= 15) & (egfr < 30)] = 3
    cls[egfr < 15] = 4

    return cls
    



In [6]:
egfr_true = compute_egfr_tensor(y_train, age_train, sex_train)
true_cls_train = assign_class_tensor(egfr_true).cpu().numpy()

egfr_true = compute_egfr_tensor(y_val, age_val, sex_val)
true_cls_val = assign_class_tensor(egfr_true).cpu().numpy()

egfr_true = compute_egfr_tensor(y_test, age_test, sex_test)
true_cls_test = assign_class_tensor(egfr_true).cpu().numpy()

y_train = true_cls_train
y_val = true_cls_val
y_test = true_cls_test

## Data flattening

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, logging, math, os, json
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger(__name__)


# PPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPP
# 2. XGBoost  SHARED MODEL ACROSS ALL TIME STEPS
# PPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPP

def build_flat_dataset(X_aug: np.ndarray, y: np.ndarray):
    """
    Flatten (N, 7, D)  (N*7, D+1) by appending a 'day_index' feature.
    Labels are repeated for each time step.
    Returns X_flat (N*7, D+1), y_flat (N*7,), sample_ids (N*7,)
    """
    N, T, D = X_aug.shape

    X_with_day = X_aug
                                 
    X_flat = X_with_day.reshape(N, D * T)
    
    y_flat = y

    sample_ids = np.repeat(np.arange(N), T)

    return X_flat.astype(np.float32), y_flat.astype(int), sample_ids


print(f"X_train: {X_train.shape}")

X_tr_aug = X_train
X_val_aug = X_val
X_te_aug = X_test
    
print(f"X_tr_aug: {X_tr_aug.shape}")
        #  Step 3: Train shared XGBoost 
        #log.info("Step 2: Training shared XGBoost &")
X_flat_tr, y_flat_tr, _   = build_flat_dataset(X_tr_aug, y_train)
X_flat_val, y_flat_val, _ = build_flat_dataset(X_val_aug, y_val)
X_flat_test, y_flat_test, _ = build_flat_dataset(X_te_aug, y_test)
print(f"X_tr_aug: {X_flat_tr.shape}")


X_train: (47935, 7, 166)
X_tr_aug: (47935, 7, 166)
X_tr_aug: (47935, 1162)


# Remove Zero Columns 

In [9]:
s = np.sum(X_flat_tr, axis=0)
print(f"s.shape: {s.shape}")
#print(f"s: {s}")
nz_indx = np.where(s > 0)[0]
print(f"indx length: {len(nz_indx)}")
#indx = np.where(s<=100)[0]
#print(f"indx: {indx}")


s.shape: (1162,)
indx length: 761


In [10]:
print(f"X_flat_tr size: {X_flat_tr.shape}")
X_flat_tr = X_flat_tr[:, nz_indx]
X_flat_val = X_flat_val[:, nz_indx]
X_flat_test = X_flat_test[:, nz_indx]
print(f"X_flat_tr size: {X_flat_tr.shape}")


X_flat_tr size: (47935, 1162)
X_flat_tr size: (47935, 761)


In [11]:
s = np.sum(X_tr_aug, axis=1)
s = np.sum(s, axis=0)
print(f"s.shape: {s.shape}")
#print(f"s: {s}")
nz_indx = np.where(s > 0)[0]
print(f"indx length: {len(nz_indx)}")
#indx = np.where(s<=100)[0]
#print(f"indx: {indx}")


s.shape: (166,)
indx length: 110


In [12]:
print(f"X_tr_aug size: {X_tr_aug.shape}")
X_tr_aug = X_tr_aug[:, :, nz_indx]
X_val_aug = X_val_aug[:, :, nz_indx]
X_te_aug = X_te_aug[:, :, nz_indx]
print(f"X_tr_aug size: {X_tr_aug.shape}")


X_tr_aug size: (47935, 7, 166)
X_tr_aug size: (47935, 7, 110)


## Data loader

In [16]:
from torch.utils.data import DataLoader, TensorDataset


X_train_t = torch.as_tensor(X_tr_aug, dtype=torch.float32)
y_train_t = torch.as_tensor(y_train, dtype=torch.float32).long() 
X_val_t = torch.as_tensor(X_val_aug, dtype=torch.float32)
y_val_t = torch.as_tensor(y_val, dtype=torch.float32).long() 
X_test_t = torch.as_tensor(X_te_aug, dtype=torch.float32)
y_test_t = torch.as_tensor(y_test, dtype=torch.float32).long() 

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

batch_size = 1*1024

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers= 0, pin_memory=True, persistent_workers=False)
val_loader  =  DataLoader(val_dataset,  batch_size=batch_size, shuffle=False, num_workers= 0, pin_memory=True, persistent_workers=False)
test_loader =  DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers= 0, pin_memory=True, persistent_workers=False)